## 05 — Temporal Alignment and Data Aggregation

In [ ]:
### Align observations to the target event and prepare time bins

In [64]:
import pandas as pd
from pathlib import Path


In [66]:
#Get the main project folder (moves up from 'notebooks' to root) ->Python_eye_tracking_2x2
project_folder = Path.cwd().parent



In [68]:
# Define project path and file location
csv_file_path = project_folder / "data" / "interim" / "et_data_transformed_04.csv"

In [70]:
# Load the dataset back into pandas
et_data_transformed_04 = pd.read_csv(csv_file_path)

In [71]:
et_data_transformed_04.head()

,ParticipantName,items,Lists,RecordingTimestamp,time_diff,GazeEventDur_ms,Validity left,Validity right,EyeMovementType,GazePointX,...,sentences,left,right,out,Condition,AOI1,AOI2,Subject,Object,ROI
0,part05,Item12c,List2,139899,0,225,Valid,Valid,Fixation,477.0,...,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS,Subject,Object,1,0,Subject
1,part05,Item12c,List2,139907,8,225,Valid,Valid,Fixation,482.0,...,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS,Subject,Object,1,0,Subject
2,part05,Item12c,List2,139915,16,225,Valid,Valid,Fixation,476.0,...,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS,Subject,Object,1,0,Subject
3,part05,Item12c,List2,139924,25,225,Valid,Valid,Fixation,485.0,...,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS,Subject,Object,1,0,Subject
4,part05,Item12c,List2,139932,33,225,Valid,Valid,Fixation,495.0,...,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS,Subject,Object,1,0,Subject


In [72]:
# Inspect the value distribution
print(et_data_transformed_04["ROI"].value_counts()) #ROI = Region of Interest

ROI
out        213319
Subject     84218
Object      73632
Name: count, dtype: int64


In [73]:
#Locate project root and metadata folder path
project_folder = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
interim_folder = project_folder / "data" / "interim"

# Define the path to your timing metadata Excel file
excel_file_path = interim_folder / "TimingInfo1000msbeforPron.xlsx"  # or

# 3. Read the metadata into pandas
timing_info = pd.read_excel(excel_file_path)

In [78]:
timing_info.head()

,PresentedMediaName,items,Tense,ms_onset,WindowEndTime
0,Item01a.avi,Item01a,svo_sp,6847,9657
1,Item01b.avi,Item01b,svo_op,6931,10063
2,Item01c.avi,Item01c,ovs_sp,6755,9831
3,Item01d.avi,Item01d,ovs_op,6746,9747
4,Item02a.avi,Item02a,svo_sp,5413,8673


In [80]:
#Merge metadata to get 'ms_onset' for each item
et_filtered = et_data_transformed_04.merge(
    timing_info[["items", "ms_onset"]], 
    on="items", 
    how="left"
)

In [82]:
#Keep only rows where raw ms is greater than or equal to ms_onset
et_filtered = et_filtered[et_filtered["ms"] >= et_filtered["ms_onset"]]

In [84]:
et_filtered.head()

,ParticipantName,items,Lists,RecordingTimestamp,time_diff,GazeEventDur_ms,Validity left,Validity right,EyeMovementType,GazePointX,...,left,right,out,Condition,AOI1,AOI2,Subject,Object,ROI,ms_onset
425,part05,Item12c,List2,147132,7233,83,Valid,Valid,Fixation,756.0,...,0,0,2,OVS,Subject,Object,0,0,out,7003
426,part05,Item12c,List2,147141,7242,83,Valid,Valid,Fixation,758.0,...,0,0,2,OVS,Subject,Object,0,0,out,7003
427,part05,Item12c,List2,147149,7250,83,Valid,Valid,Fixation,760.0,...,0,0,2,OVS,Subject,Object,0,0,out,7003
428,part05,Item12c,List2,147157,7258,83,Valid,Valid,Fixation,757.0,...,0,0,2,OVS,Subject,Object,0,0,out,7003
429,part05,Item12c,List2,147166,7267,83,Invalid,Valid,Fixation,762.0,...,0,0,2,OVS,Subject,Object,0,0,out,7003


In [86]:
# (Optional) Re-zero 'ms' so onset becomes time = 0
et_filtered["ms_rel"] = et_filtered["ms"] - et_filtered["ms_onset"]
et_filtered["bin20_rel"] = (et_filtered["ms_rel"] // 20) * 20

In [88]:
# Create an independent copy of et_data_joined
et_data_aggregated_05 = et_filtered.copy()

In [90]:
# Define the interim output path
interim_folder = project_folder / "data" / "interim"
# Export your current DataFrame to CSV

csv_file_path = interim_folder / "et_data_aggregated_05.csv"
et_data_aggregated_05.to_csv(csv_file_path, index=False)

In [91]:
# End of the aggregation process, editing and checking the eye-tracking file with the metadata _ time onset. 
# The next step is to visualize the data for descriptive analysis.